## Проверка data-pipeline


In [1]:
import os
import sys

import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

from collections import Counter
import pandas as pd

PROJECT_ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.dataloaders import create_dataloaders, create_test_dataloader


In [3]:
# Создаём DataLoader на основе processed CSV и локальных изображений
train_loader, val_loader = create_dataloaders(
    batch_size=32,
    num_workers=0,
    image_size=224,
    use_weighted_sampling=True
)

test_loader = create_test_dataloader(
    batch_size=32,
    num_workers=0,
    image_size=224
)

print("train объектов:", len(train_loader.dataset))
print("val объектов:", len(val_loader.dataset))
print("test объектов для предсказания:", len(test_loader.dataset))


train объектов: 4311
val объектов: 477
test объектов для предсказания: 47791


In [4]:
# Берём первый batch из train и test
images, targets = next(iter(train_loader))
test_images, test_image_ids, test_item_ids = next(iter(test_loader))


In [5]:
# Проверяем формы batch
print("train images:", images.shape)
print("train targets:", targets.shape)
print("test images:", test_images.shape)
print("первые image_id из test:", list(test_image_ids[:5]))


train images: torch.Size([32, 3, 224, 224])
train targets: torch.Size([32])
test images: torch.Size([32, 3, 224, 224])
первые image_id из test: ['15822317371', '15948569108', '16030687230', '16033186338', '15430455676']


In [6]:
# Загружаем ResNet-18 с предобученными весами
model = resnet18(weights=ResNet18_Weights.DEFAULT)


In [7]:
# Меняем последний слой под наши 19 классов
num_classes = 19
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)


In [8]:
# Прогоняем batch через модель без обучения
model.eval()

with torch.no_grad():
    train_outputs = model(images)
    test_outputs = model(test_images)

print("train outputs:", train_outputs.shape)
print("test outputs:", test_outputs.shape)


train outputs: torch.Size([32, 19])
test outputs: torch.Size([32, 19])


In [9]:
# прверка балансировки
def check_loader_class_balance(loader, max_batches=None):
    class_counter = Counter()

    for batch_idx, (_, targets) in enumerate(loader):
        class_counter.update(targets.tolist())

        if max_batches is not None and batch_idx + 1 >= max_batches:
            break

    balance_df = (
        pd.DataFrame(
            class_counter.items(),
            columns=["class_id", "count"]
        )
        .sort_values("class_id")
        .reset_index(drop=True)
    )

    balance_df["ratio"] = balance_df["count"] / balance_df["count"].sum()

    return balance_df

In [10]:
train_balance_df = check_loader_class_balance(train_loader)

train_balance_df

,class_id,count,ratio
0,0,215,0.049872
1,1,222,0.051496
2,2,205,0.047553
3,3,241,0.055904
4,4,235,0.054512
5,5,232,0.053816
6,6,221,0.051264
7,7,229,0.053120
8,8,205,0.047553
9,9,215,0.049872
